In [ ]:
%pip install sympy

105.81s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


In [2]:
import sympy as sp
from IPython.display import display, Math

# $\textbf{A Small "Engine" for tensors in General Relativity }$ 
$\text{(Specialized for the calculation of various tensors for the Problem of Magnetohydrodynamics)}$
$\text{The calculations are done in symbolic fashion, and later work needs to be done to incorporate these effectively.}$ 

$\text{This is the coordinates that we are working with}$
$$ (t,\, r,\, \theta,\, \phi) $$

In [3]:
# defining the Coordinate system in python
t,r,th,ph = sp.symbols('t r theta phi')
N = sp.Function('N')(r,th)
A = sp.Function('A')(r,th)
B = sp.Function('B')(r,th)
coords=[t,r,th,ph]

$\text{Similarly we define the space-time element}$
$$ ds^2 = - N^2 dt^2 + A^2 dr^2 + A^2 r^2 d\theta^2 + B^2 r^2 \sin\theta^2 d\phi^2$$
$\text{and so work with the following metric and inverse metric:}$

$$ g_{\mu\nu}
= \begin{pmatrix}
-N^2 & 0 & 0 & 0 \\
0 & A^2 & 0 & 0 \\
0 & 0 & A^2r^2 & 0 \\
0 & 0 & 0 & B^2 r^2\sin^2\theta
\end{pmatrix} \;\; g^{\mu\nu}
= \begin{pmatrix}
-N^{-2} & 0 & 0 & 0 \\
0 & A^{-2} & 0 & 0 \\
0 & 0 & A^{-2}r^{-2} & 0 \\
0 & 0 & 0 & B^{-2} r^{-2}\sin^{-2}\theta
\end{pmatrix}\,.$$

In [4]:
# metric defined using diag
sin=sp.sin
g=sp.diag(-N**2, A**2, A**2*r**2, B**2*r**2*sin(th)**2)
#inverse metric
ginv=g.inv()
# dimension 
dim=4

$\text{Christoffel symbols}$ 
$$ \Gamma^{\alpha}{}_{\beta\gamma} = \frac{1}{2} g^{\alpha\delta} \left( \partial_{\beta} g_{\delta\gamma} + \partial_{\gamma} g_{\delta\beta} - \partial_{\delta} g_{\beta\gamma} \right)$$

$\text{The code below calcualtes the nonzero components of the Christoffel symbols and then prints them.}$
$\text{ (along with the count of the nonzero components)}$

In [5]:
# Christoffel symbols
Gamma=[[[0 for k in range(dim)] for j in range(dim)] for i in range(dim)]
for a in range(dim):
  for b in range(dim):
    for c in range(dim):
      expr=0
      for d in range(dim):
        expr += ginv[a,d]*(sp.diff(g[d,c], coords[b]) + sp.diff(g[d,b], coords[c]) - sp.diff(g[b,c], coords[d]))
      Gamma[a][b][c]=sp.simplify(expr/2)
# nonzero christoffel
names=['t','r','theta','phi']
nonG=[]
for a in range(dim):
  for b in range(dim):
    for c in range(b,dim):
      if Gamma[a][b][c]!=0:
        nonG.append((names[a],names[b],names[c],sp.factor(Gamma[a][b][c])))
print('Christoffel count', len(nonG))
for item in nonG:
 print(item)

Christoffel count 14
('t', 't', 'r', Derivative(N(r, theta), r)/N(r, theta))
('t', 't', 'theta', Derivative(N(r, theta), theta)/N(r, theta))
('r', 't', 't', N(r, theta)*Derivative(N(r, theta), r)/A(r, theta)**2)
('r', 'r', 'r', Derivative(A(r, theta), r)/A(r, theta))
('r', 'r', 'theta', Derivative(A(r, theta), theta)/A(r, theta))
('r', 'theta', 'theta', -r*(r*Derivative(A(r, theta), r) + A(r, theta))/A(r, theta))
('r', 'phi', 'phi', -r*(r*Derivative(B(r, theta), r) + B(r, theta))*B(r, theta)*sin(theta)**2/A(r, theta)**2)
('theta', 't', 't', N(r, theta)*Derivative(N(r, theta), theta)/(r**2*A(r, theta)**2))
('theta', 'r', 'r', -Derivative(A(r, theta), theta)/(r**2*A(r, theta)))
('theta', 'r', 'theta', (r*Derivative(A(r, theta), r) + A(r, theta))/(r*A(r, theta)))
('theta', 'theta', 'theta', Derivative(A(r, theta), theta)/A(r, theta))
('theta', 'phi', 'phi', -(B(r, theta)*sin(2*theta) - cos(2*theta)*Derivative(B(r, theta), theta) + Derivative(B(r, theta), theta))*B(r, theta)/(2*A(r, theta)

$\text{Ricci Tensor}$
$$ R_{\mu\nu} =
\partial_{\lambda}\Gamma^{\lambda}{}_{\mu\nu} -
\partial_{\nu}\Gamma^{\lambda}{}_{\mu\lambda} +
\Gamma^{\lambda}{}_{\mu\nu}\Gamma^{\sigma}{}_{\lambda\sigma} -
\Gamma^{\sigma}{}_{\mu\lambda}\Gamma^{\lambda}{}_{\nu\sigma} $$
$\text{Like the Crystoffel symbols, and the rest are similar.}$

In [6]:
# Ricci Tensor 
Ric=[[0 for j in range(dim)] for i in range(dim)]
for mu in range(dim):
  for nu in range(dim):
    expr=0
    for lam in range(dim):
      expr += sp.diff(Gamma[lam][mu][nu], coords[lam]) - sp.diff(Gamma[lam][mu][lam], coords[nu])
      for sig in range(dim):
        expr += Gamma[lam][mu][nu]*Gamma[sig][lam][sig] - Gamma[sig][mu][lam]*Gamma[lam][nu][sig]
    Ric[mu][nu]=sp.factor(sp.simplify(expr))
print('\nRicci nonzero')
for i in range(dim):
 for j in range(i,dim):
  if Ric[i][j]!=0:
    print(names[i],names[j], sp.factor(Ric[i][j]))


Ricci nonzero
t t (r**2*B(r, theta)*tan(theta)*Derivative(N(r, theta), (r, 2)) + r**2*tan(theta)*Derivative(B(r, theta), r)*Derivative(N(r, theta), r) + 2*r*B(r, theta)*tan(theta)*Derivative(N(r, theta), r) + B(r, theta)*tan(theta)*Derivative(N(r, theta), (theta, 2)) + B(r, theta)*Derivative(N(r, theta), theta) + tan(theta)*Derivative(B(r, theta), theta)*Derivative(N(r, theta), theta))*N(r, theta)/(r**2*A(r, theta)**2*B(r, theta)*tan(theta))
r r -(r**2*A(r, theta)**2*B(r, theta)*tan(theta)*Derivative(N(r, theta), (r, 2)) + r**2*A(r, theta)**2*N(r, theta)*tan(theta)*Derivative(B(r, theta), (r, 2)) + r**2*A(r, theta)*B(r, theta)*N(r, theta)*tan(theta)*Derivative(A(r, theta), (r, 2)) - r**2*A(r, theta)*B(r, theta)*tan(theta)*Derivative(A(r, theta), r)*Derivative(N(r, theta), r) - r**2*A(r, theta)*N(r, theta)*tan(theta)*Derivative(A(r, theta), r)*Derivative(B(r, theta), r) - r**2*B(r, theta)*N(r, theta)*tan(theta)*Derivative(A(r, theta), r)**2 + 2*r*A(r, theta)**2*N(r, theta)*tan(theta)*D

$\text{Ricci Scalar}$ 
$$ R = g^{\mu\nu} R_{\mu\nu} $$ 

In [7]:
# Ricci Scalar
Rscalar=sp.factor(sp.simplify(sum(ginv[i,j]*Ric[i][j] for i in range(dim) for j in range(dim))))
print('\nRscalar=', Rscalar)


Rscalar= -2*(r**2*A(r, theta)**2*B(r, theta)*sin(2*theta)*Derivative(N(r, theta), (r, 2)) + r**2*A(r, theta)**2*N(r, theta)*sin(2*theta)*Derivative(B(r, theta), (r, 2)) + r**2*A(r, theta)**2*sin(2*theta)*Derivative(B(r, theta), r)*Derivative(N(r, theta), r) + r**2*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*Derivative(A(r, theta), (r, 2)) - r**2*B(r, theta)*N(r, theta)*sin(2*theta)*Derivative(A(r, theta), r)**2 + 2*r*A(r, theta)**2*B(r, theta)*sin(2*theta)*Derivative(N(r, theta), r) + 3*r*A(r, theta)**2*N(r, theta)*sin(2*theta)*Derivative(B(r, theta), r) + r*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*Derivative(A(r, theta), r) - A(r, theta)**2*B(r, theta)*sin(2*theta)*tan(theta)*Derivative(N(r, theta), theta) + A(r, theta)**2*B(r, theta)*sin(2*theta)*Derivative(N(r, theta), (theta, 2)) + 2*A(r, theta)**2*B(r, theta)*Derivative(N(r, theta), theta) - 2*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), theta) + A(r, theta)**2*N(r, theta)*sin(2*theta)*D

$\text{Einstein's Tensor}$
$$ G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2} g_{\mu\nu} R $$
$$ G^{\mu}{}_{\nu} = g^{\mu\alpha} G_{\alpha\nu}.$$

In [8]:
# Einstein's Tensor
Ein=[[sp.factor(sp.simplify(Ric[i][j]-sp.Rational(1,2)*g[i,j]*Rscalar)) for j in range(dim)] for i in range(dim)]
print('\nEinstein nonzero cov')
for i in range(dim):
 for j in range(i,dim):
  if Ein[i][j]!=0:
    print(names[i],names[j], sp.factor(Ein[i][j]))


Einstein nonzero cov
t t -(r**2*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), (r, 2)) + r**2*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), (r, 2)) - r**2*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), r)**2 + 3*r*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), r) + r*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), r) - A(r, theta)**2*B(r, theta)*sin(2*theta)*tan(theta)**2*Derivative(N(r, theta), theta) - A(r, theta)**2*B(r, theta)*sin(2*theta)*Derivative(N(r, theta), theta) + 2*A(r, theta)**2*B(r, theta)*tan(theta)*Derivative(N(r, theta), theta) - 2*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)**2*Derivative(B(r, theta), theta) + A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), (theta, 2)) + 4*A(r, theta)**2*N(r, theta)*tan(theta)*Derivative(B(r, theta), theta) + A(r, theta)*B(r, theta)*N(r, theta)*sin

In [9]:
# mixed components
Gmix=[[sp.factor(sp.simplify(sum(ginv[i,k]*Ein[k][j] for k in range(dim)))) for j in range(dim)] for i in range(dim)]
print('\nEinstein mixed nonzero')
for i in range(dim):
 for j in range(dim):
  if Gmix[i][j]!=0:
    print(names[i],names[j], sp.factor(Gmix[i][j]))


Einstein mixed nonzero
t t (r**2*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), (r, 2)) + r**2*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), (r, 2)) - r**2*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), r)**2 + 3*r*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), r) + r*A(r, theta)*B(r, theta)*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(A(r, theta), r) - A(r, theta)**2*B(r, theta)*sin(2*theta)*tan(theta)**2*Derivative(N(r, theta), theta) - A(r, theta)**2*B(r, theta)*sin(2*theta)*Derivative(N(r, theta), theta) + 2*A(r, theta)**2*B(r, theta)*tan(theta)*Derivative(N(r, theta), theta) - 2*A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)**2*Derivative(B(r, theta), theta) + A(r, theta)**2*N(r, theta)*sin(2*theta)*tan(theta)*Derivative(B(r, theta), (theta, 2)) + 4*A(r, theta)**2*N(r, theta)*tan(theta)*Derivative(B(r, theta), theta) + A(r, theta)*B(r, theta)*N(r, theta)*si

$\text{This Pile of code just writes a Latex version of the results previously calculated, these are the results that we have used in the article, from which you have been probably lead to here (assuming; non-controversially; that you are)}$

In [10]:
sp.init_printing()
latex_names = ['t', 'r', r'\theta', r'\phi']
# nonzero christoffel
display(Math(r"\text{Nonzero Christoffel symbols}"))
for a in range(dim):
    for b in range(dim):
        for c in range(b, dim):
            if Gamma[a][b][c] != 0:
                expr = sp.factor(Gamma[a][b][c])
                display(Math(
                    rf"\Gamma^{{{latex_names[a]}}}_{{{latex_names[b]}{latex_names[c]}}} = {sp.latex(expr)}"
                ))

# Ricci tensor
display(Math(r"\text{Nonzero Ricci tensor components}"))
for i in range(dim):
    for j in range(i, dim):
        if Ric[i][j] != 0:
            expr = sp.factor(Ric[i][j])
            display(Math(
                rf"R_{{{latex_names[i]}{latex_names[j]}}} = {sp.latex(expr)}"
            ))

# Ricci scalar
display(Math(r"R = " + sp.latex(Rscalar)))

# Einstein tensor covariant
display(Math(r"\text{Nonzero Einstein tensor components } G_{\mu\nu}"))
for i in range(dim):
    for j in range(i, dim):
        if Ein[i][j] != 0:
            expr = sp.factor(Ein[i][j])
            display(Math(
                rf"G_{{{latex_names[i]}{latex_names[j]}}} = {sp.latex(expr)}"
            ))

# Einstein tensor mixed
display(Math(r"\text{Nonzero mixed Einstein tensor components } G^\mu{}_\nu"))
for i in range(dim):
    for j in range(dim):
        if Gmix[i][j] != 0:
            expr = sp.factor(Gmix[i][j])
            display(Math(
                rf"G^{{{latex_names[i]}}}_{{{latex_names[j]}}} = {sp.latex(expr)}"
            ))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>